# Código de treino do modelo de decisão de protocolo

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from joblib import dump, load

import logging

np.random.seed(42)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Load and preprocess dataset
try:
    df = pd.read_csv('./appliances_energy/energydata_complete.csv')
    logging.info("Dataset loaded successfully. Shape: %s", df.shape)
except FileNotFoundError:
    logging.error("Dataset 'hybrid.csv' not found.")
    sys.exit(1)

# Feature Engineering
#def generate_context_features(df):
#    df['battery'] = df['TCP_w_size'].apply(lambda x: min(100, (x / 29480) * 100))
#    # Simulate cpu_usage and memory_usage based on packet size and entropy
#    df['cpu_usage'] = df['Pck_size'].apply(lambda x: min(90, max(10, (x / 765) * 50 + np.random.normal(0, 5))))
#    df['memory_usage'] = df['Entropy'].apply(lambda x: min(90, max(10, (x / 5.59) * 60 + np.random.normal(0, 5))))
#    df['data_size'] = df['Pck_size'] / 765
#    df['sensitivity'] = (df['Entropy'] / 5.59) * 0.6 + (df['Pck_size'] / 765) * 0.4
#    df['sensitivity'] = df['sensitivity'].clip(0, 1)
#    df['data_type'] = df[['HTTP', 'UDP']].apply(
#        lambda x: 'video' if x['HTTP'] > 0 else ('telemetry' if x['UDP'] > 0 else 'other'), axis=1
#    )
#    df['data_type'] = LabelEncoder().fit_transform(df['data_type'])  # Encode categorical feature
#    df['bandwidth'] = df['IP_add_count'].apply(lambda x: min(100, x * 20))
#    df['latency'] = df['Portcl_src'].apply(lambda x: min(50, x * 10))
#    df['congestion'] = df['Portcl_dst'].apply(lambda x: min(1, x / 10))
#    return df

#def generate_context_features(df):
#    df['sensitivity'] = random.uniform(1, 3)  
#    return df
#
#df = generate_context_features(df)
#context_features = ['battery', 'cpu_usage', 'memory_usage', 'data_size', 'data_type', 'bandwidth', 'latency', 'congestion'] #'sensitivity'
context_features = df.columns.tolist()

df = pd.DataFrame(df)
#df.to_csv('../dataset/df_processed_original.csv', index=False)

In [ ]:
def generate_synthetic_labels(df):
    e_alg = []
    for _, row in df.iterrows():
        sens_j = row['sensitivity']
        if sens_j == 1:
            e_alg.append('x25519')
        elif sens_j == 2:
            e_alg.append('mlkem768')
        else:
            e_alg.append('X25519MLKEM768')
    return pd.DataFrame({'e_alg': e_alg})

# Treino

In [ ]:
y_df = generate_synthetic_labels(df)
e_alg_encoder = LabelEncoder()
y_df['e_alg'] = e_alg_encoder.fit_transform(y_df['e_alg'])

X = df[context_features]
y = y_df['e_alg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y['e_alg'])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_scaled, y_train_e_alg = smote.fit_resample(X_train_scaled, y_train['e_alg'])
y_train_k_freq = np.repeat(y_train['k_freq'].values, X_train_scaled.shape[0] // X_train.shape[0], axis=0)
if len(y_train_k_freq) < X_train_scaled.shape[0]:
    y_train_k_freq = np.pad(y_train_k_freq, (0, X_train_scaled.shape[0] - len(y_train_k_freq)), mode='edge')
y_train = pd.DataFrame({'e_alg': y_train_e_alg, 'k_freq': y_train_k_freq})

# Hyperparameter tuning for models
param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1, 0.3]}
classifier = GridSearchCV(XGBClassifier(random_state=42, eval_metric='mlogloss'), param_grid, cv=5, n_jobs=-1)
#regressor = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=5, n_jobs=-1)
classifier.fit(X_train_scaled, y_train['e_alg'])
#regressor.fit(X_train_scaled, y_train['k_freq'])
logging.info("Best classifier params: %s", classifier.best_params_)
#logging.info("Best regressor params: %s", regressor.best_params_)
dump(classifier.best_estimator_, '../models/context_classifier.joblib')
#dump(regressor.best_estimator_, '../models/context_regressor.joblib')

# Avaliação

In [ ]:
# Evaluate
y_pred_e_alg = classifier.predict(X_test_scaled)

print("\nEncryption Algorithm Classification Report:")
print(classification_report(y_test['e_alg'], y_pred_e_alg, target_names=e_alg_encoder.classes_))


# Graphs
cm_e_alg = confusion_matrix(y_test['e_alg'], y_pred_e_alg)
cm_percentage = cm_e_alg / cm_e_alg.sum(axis=1)[:, np.newaxis] * 100
plt.figure(figsize=(10, 8))
sns.heatmap(cm_percentage, annot=True, fmt=".2f", cmap='coolwarm',
            xticklabels=e_alg_encoder.classes_, yticklabels=e_alg_encoder.classes_)
plt.title('Encryption Algorithm Confusion Matrix (%)')
plt.xlabel('Predicted')
plt.ylabel('True')
#plt.savefig("../models/figs/e_alg_confusion.pdf")
plt.show()

feature_importance = pd.DataFrame({
    'Feature': context_features,
    'Importance_e_alg': classifier.best_estimator_.feature_importances_,
}).melt(id_vars='Feature', var_name='Target', value_name='Importance')
plt.figure(figsize=(12, 6))
sns.barplot(x='Importance', y='Feature', hue='Target', data=feature_importance)
plt.title('Feature Importance for Encryption Algorithm and Key Rotation')
#plt.savefig("../models/figs/feature_importance.pdf")
plt.show()

e_alg_counts = pd.Series(e_alg_encoder.inverse_transform(y_pred_e_alg)).value_counts()
plt.figure(figsize=(8, 6))
e_alg_counts.plot(kind='bar', color=['red', 'green', 'blue'])
plt.title('Predicted Encryption Algorithm Distribution')
plt.xlabel('Encryption Algorithm')
plt.ylabel('Count')
#plt.savefig("../models/figs/e_alg_distribution.pdf")
plt.show()